In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from google.cloud import storage
from google.cloud import bigquery
import joblib
import json
import tempfile
import io




project_id = "project-3e6b348d-e2ae-4a47-9af"
bucket_name = "project-3e6b348d-e2ae-4a47-9af_cloudbuild"

DATASET = "near_earth_monitoring"
TABLE = "inference_events"

# ===============================================
def load_pkl_from_gcs(
        project_id: str,
        bucket_name: str,
        blob_name: str
) -> pd.DataFrame:
    client = storage.Client(project=project_id)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    with tempfile.TemporaryDirectory() as tempdir:
        local_path = Path(tempdir) / Path(blob_name).name
        blob.download_to_filename(local_path)
        return joblib.load(local_path)
     
# ===============================================
def parse_json_values(value):
    if isinstance(value, str):
        return json.loads(value)
    return value
# ==============================================   

reference_pha = load_pkl_from_gcs(
    project_id = project_id,
    bucket_name = bucket_name,
    blob_name = "artifacts/X_train_pha.pkl"
)

y_train_pha = load_pkl_from_gcs(
    project_id = project_id,
    bucket_name = bucket_name,
    blob_name = "artifacts/y_train_pha.pkl"
)

client = bigquery.Client(project=project_id)

query = f"""
    SELECT
        timestamp,
        model_name,
        model_version,
        endpoint,
        features,
        status,
        latency_ms,
        prediction,
        error
    FROM `{project_id}.{DATASET}.{TABLE}`
    WHERE model_name=@model_name
    AND status = 'success'
    AND timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL @lookback_hours HOUR)
    ORDER BY timestamp DESC;
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("model_name", "STRING", "pha"),
        bigquery.ScalarQueryParameter("lookback_hours", "INT64", 24 * 14)
    ]
)


query_job = client.query(query, job_config=job_config)

current_pha = query_job.to_dataframe()
current_pha_target = current_pha['prediction']
current_pha = pd.json_normalize(current_pha['features'].apply(parse_json_values))


In [2]:
reference_pha.reset_index(drop=True, inplace=True)



In [3]:
reference_moid = load_pkl_from_gcs(
    project_id=project_id,
    bucket_name=bucket_name,
    blob_name="artifacts/X_train_moid.pkl"
)

reference_moid.head()

y_train_moid = load_pkl_from_gcs(
    project_id=project_id,
    bucket_name=bucket_name,
    blob_name="artifacts/y_train_moid.pkl"
)


query_moid = f"""
    SELECT 
        timestamp,
        model_name,
        model_version,
        endpoint,
        features,
        status,
        latency_ms,
        prediction,
        error
    FROM `{project_id}.{DATASET}.inference_events_moid`
    WHERE model_name=@model_name
    AND status = 'success'
    AND timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL @lookback_hours HOUR)
    ORDER BY timestamp DESC;

"""

job_config_moid = bigquery.QueryJobConfig(
    query_parameters = [
        bigquery.ScalarQueryParameter("model_name", "STRING", "moid"),
        bigquery.ScalarQueryParameter("lookback_hours", "INT64", 24*15)
    ]
)

query_job_moid = client.query(query_moid, job_config_moid)

current_moid = query_job_moid.to_dataframe()
current_moid_target = current_moid['prediction']
current_moid = pd.json_normalize(current_moid['features'].apply(parse_json_values))
current_moid

,H,aphelion_distance_au,class_code,condition_code,data_arc,diameter_km,eccentricity,inclination_deg,mean_motion_deg_day,orbital_period_days,perihelion_distance_au,pha,semimajor_axis_au,size_category
0,33.183,729.869,APO,8,23579.859,41.924,0.479,87.620,2.145,60584.848,0.226,0,15.623,23.728
1,25.128,433.281,APO,1,47095.770,41.998,0.500,71.508,1.752,2427959.161,0.271,0,378.676,19.828
2,16.246,65.209,APO,8,12776.633,10.702,0.588,134.156,0.639,1933214.213,0.115,1,152.154,26.732
3,17.964,234.954,APO,8,20238.919,16.216,0.472,88.436,1.995,1141238.807,0.316,0,127.449,26.705
4,29.369,578.237,APO,1,44969.005,18.112,0.102,78.265,3.122,2039444.060,0.211,0,343.484,20.665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434,12.193,50.896,APO,8,45813.754,9.403,1.005,165.926,2.352,1823081.084,0.787,0,265.032,30.308
435,33.516,458.794,APO,1,20473.354,16.581,0.826,124.383,1.614,1544146.373,1.160,1,278.169,21.725
436,21.034,495.568,APO,8,18176.287,22.930,0.706,181.650,1.369,1619610.813,1.014,0,276.811,22.864
437,30.051,792.729,APO,1,41554.644,24.991,1.099,80.897,3.534,394176.936,0.929,0,72.772,26.868


In [4]:
numerical_columns_pha = reference_pha.select_dtypes(include='number').columns.tolist()
categorical_columns_pha = reference_pha.select_dtypes(include=['str','object']).columns.tolist()

numerical_columns_moid = reference_moid.select_dtypes(include="number").columns.tolist()
categorical_columns_moid = reference_moid.select_dtypes(include=['str', 'object']).columns.tolist()

In [5]:
import zipfile
import io
from datetime import datetime, time


from evidently.legacy.pipeline.column_mapping import ColumnMapping
from evidently import  Dataset, DataDefinition, BinaryClassification
from evidently.legacy.metric_preset import ClassificationPreset, RegressionPreset, TargetDriftPreset, DataDriftPreset
from evidently.metrics import RecallByLabel
from evidently.legacy.report import Report

reference_pha['target'] = y_train_pha.tolist()
reference_moid['target'] = y_train_moid.tolist()


In [6]:

# pha_model is a Randome Forest Classifier
pha_model = load_pkl_from_gcs(project_id=project_id, bucket_name=bucket_name, blob_name="artifacts/best_model_pha.pkl")
columntransformer_pha = load_pkl_from_gcs(project_id=project_id, bucket_name=bucket_name, blob_name="artifacts/columntransformer_pha.pkl")
pha_model.fit(columntransformer_pha.transform(reference_pha[numerical_columns_pha + categorical_columns_pha]), reference_pha["target"])

# regressor_moid is a Random Forest Regressor
moid_model = load_pkl_from_gcs(project_id=project_id, bucket_name=bucket_name, blob_name="artifacts/best_model_moid.pkl")
columntransformer_moid = load_pkl_from_gcs(project_id=project_id, bucket_name=bucket_name, blob_name="artifacts/columntransformer_moid.pkl")
moid_model.fit(columntransformer_moid.transform(reference_moid[numerical_columns_moid + categorical_columns_moid]), reference_moid["target"])

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [7]:
reference_prediction_pha =  pha_model.predict(columntransformer_pha.transform(reference_pha[numerical_columns_pha + categorical_columns_pha]))
current_prediction_pha = pha_model.predict(columntransformer_pha.transform(current_pha[numerical_columns_pha + categorical_columns_pha]))

reference_prediction_moid = moid_model.predict(columntransformer_moid.transform(reference_moid[numerical_columns_moid + categorical_columns_moid]))
current_prediction_moid = moid_model.predict(columntransformer_moid.transform(current_moid[numerical_columns_moid + categorical_columns_moid]))


reference_pha['prediction'] = reference_prediction_pha
current_pha['target'] = current_pha_target.astype(bool).tolist()
current_pha['prediction'] = current_prediction_pha


reference_moid['prediction'] = reference_prediction_moid
current_moid['target'] = current_moid_target.apply(lambda x: float(x[1:-1])).tolist()
current_moid['prediction'] = current_prediction_moid

# To resolve the error: float has no attribute shape
reference_pha[['target','prediction']] = reference_pha[['target','prediction']].astype(int)
current_pha[['target', 'prediction']] = current_pha[['target','prediction']].astype(int)



/home/merkis/macaw_ml/near_earth_asteroid_predictor/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [8]:
column_mapping = ColumnMapping()

column_mapping.task = 'classification'
column_mapping.target = 'target'
column_mapping.prediction = 'prediction'
column_mapping.pos_label = None
column_mapping.numerical_features = numerical_columns_pha
column_mapping.categorical_features = categorical_columns_pha






In [9]:
categorical_performance = Report(metrics=[DataDriftPreset()],)
categorical_performance.run(current_data=current_pha,\
                             reference_data=reference_pha, column_mapping=column_mapping)


In [10]:
reference_pha[['target','prediction']] = reference_pha[['target','prediction']].astype(int)
current_pha[['target', 'prediction']] = current_pha[['target','prediction']].astype(int)

In [ ]:
from datetime import date

drift_report_path = Path(f"../src/near_earth_asteroid_predictor/drift_reports/pha/{date.today()}")
drift_report_path.mkdir(parents=True, exist_ok=True)





In [12]:
categorical_performance_json = categorical_performance.json()

with open(drift_report_path / "report.json", "w") as f:
    f.write(categorical_performance_json)


In [13]:
column_mapping_moid = ColumnMapping()

column_mapping_moid.target = "target"
column_mapping_moid.prediction = "prediction"
column_mapping_moid.numerical_features = numerical_columns_moid
column_mapping_moid.categorical_features = categorical_columns_moid



regression_performance = Report(metrics=[DataDriftPreset()])
regression_performance.run(reference_data=reference_moid, current_data=current_moid,column_mapping=column_mapping_moid)

In [14]:
current_moid["pha"] = current_moid["pha"].astype(bool)


In [15]:
# Now the path with the magnitude directory exists without there being repeat 'near_earth_asteroid_predictor' directories
drift_report_path_moid = drift_report_path.parents[1] / f"moid/{date.today()}"
(drift_report_path.parents[1] / f"moid/{date.today()}").mkdir(parents=True, exist_ok=True)
print(drift_report_path_moid)


../src/near_earth_asteroid_predictor/drift_reports/moid/2026-05-29


In [16]:
regression_performance.json

with open(drift_report_path_moid/"report.json", 'w') as f:
    f.write(regression_performance.json())

In [17]:
html_path = Path("../src/near_earth_asteroid_predictor/drift_reports")
html_path.mkdir(parents=True, exist_ok=True)

# while html_path.open('w', format='utf-8') as f:
#     f.write()

categorical_performance.save_html(f"{html_path}/report_pha.html")
regression_performance.save_html(f"{html_path}/report_moid.html")